# Effective Inference — Kaggle 2×T4 пайплайн под **Qwen3-1.7B**

Цель: дообучить **Qwen3-1.7B** отвечать на школьные вопросы как большая модель. 1.7B выбрана ради **TL-безопасности** на L4 (4000 вопросов < 15 мин — с большим запасом) и простоты: fp16-веса 1.7B ≈ 3.4 ГБ грузятся на L4 24 GB без квантизации — AWQ не нужен (меньше рисков и потерь качества).

**Два пути:**
1. **FAST_SUBMIT** (ячейки 0–7, ~1 ч) — чистый SFT на эталонах → merge → оффлайн-судья → упаковка. **Главный неиспользованный рычаг**: все прежние 66–67 были на base-модели — дообученную ещё ни разу не шли. Этого хватит для первого сабмита.
2. **RUN_DISTILL** (ячейки 8+, +~1.5 ч, опционально) — self-distill (RAFT) поверх SFT. Дороже и рискованнее (local judge ≠ contest judge), запускай только если FAST вышел на плато.

**Среда:** всё на 2×T4 16GB (Turing → fp16, без bf16/FlashAttn2). SFT — DDP (копия модели на карту, ×2 throughput). Генерация/судья — data-parallel (шардинг запросов по картам, быстрее TP=2 без NVLink). Тяжёлые шаги — отдельными процессами, чтобы освобождалась VRAM.

**Перед запуском:** Settings → Accelerator = **GPU T4 ×2**, Internet = **ON**. Добавь как Kaggle Dataset папку `EffectiveInference` (с `dataset_ml_challenge.parquet` и `effinf_dev/`).

> ВАЖНО про посылку: в `source/config.py` уже выставлен `finetuned=True` — рантайм будет шлать MINIMAL_SYSTEM без few-shot, ровно как училась модель. Не меняй этого при отправке дообученной модели.

## 0. Установка зависимостей

In [ ]:
# ВАЖНО: Qwen3 поддерживается только с transformers>=4.51 и vllm>=0.8.5 — старые пины
# давали KeyError: 'qwen3'. Версии ниже взаимно совместимы и обе знают qwen3.
# autoawq НЕ нужен (1.7B шлём в fp16). bitsandbytes — только если включишь QLoRA.
!pip install -q -U vllm==0.8.5.post1 transformers==4.51.3 peft==0.14.* \
    accelerate==1.4.* "bitsandbytes>=0.46.1" datasets pyarrow
import transformers; print('transformers', transformers.__version__)
print('deps installed — ПЕРЕЗАПУСТИ ЯДРО (Run -> Restart) и запускай со следующей ячейки')

## 1. Config, рабочий каталог и helper для data-parallel генерации

In [ ]:
import os, shutil, subprocess, glob

# Автопоиск папки EffectiveInference в подключённых датасетах (слаг ≠ отображаемое имя).
def find_src():
    for parquet in glob.glob('/kaggle/input/**/dataset_ml_challenge.parquet', recursive=True):
        d = os.path.dirname(parquet)
        if os.path.isdir(os.path.join(d, 'effinf_dev')):
            return d
    for dev in glob.glob('/kaggle/input/**/effinf_dev', recursive=True):
        return os.path.dirname(dev)
    raise FileNotFoundError('Не нашёл EffectiveInference в /kaggle/input — проверь Add Input')

SRC = find_src()
WORK = '/kaggle/working/EffectiveInference'
print('SRC =', SRC)

# Копируем repo в writable working (input — read-only).
if not os.path.exists(WORK):
    shutil.copytree(SRC, WORK)
DEV = os.path.join(WORK, 'effinf_dev')
os.chdir(DEV)
print('cwd =', os.getcwd())

# --- гиперпараметры пайплайна (1.7B) ---
BASE        = 'Qwen/Qwen3-1.7B'
EPOCHS      = 2
MAX_LEN     = 2048
JUDGE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
EVAL_MAX_TOKENS = 1280   # покрывает ~95% длин эталонов; 1.7B на L4 быстрая → TL не риск

# --- опциональный self-distill (урезан, чтобы влезть в бюджет) ---
RUN_DISTILL   = False  # FAST_SUBMIT по умолчанию; поставь True для RAFT-прохода
DISTILL_LIMIT = 1500   # было 4000 — урезано: distill даёт убывающую отдачу
DISTILL_N     = 3      # было 6 — урезано

os.makedirs('work', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def gen_distill_dp(model_dir, out, n, limit, max_tokens=1280, num_gpus=2):
    """Data-parallel генерация: num_gpus процессов, по одной карте на каждый
    (CUDA_VISIBLE_DEVICES), запросы шардятся, шарды склеиваются. Быстрее TP=2 на
    T4 без NVLink. Каждый процесс на выходе освобождает VRAM."""
    shards, procs = [], []
    for g in range(num_gpus):
        sh = f'{out}.shard{g}'
        shards.append(sh)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        cmd = (f'python gen_distill.py --model_dir {model_dir} --train_jsonl data/train_minimal.jsonl '
               f'--out {sh} --n {n} --limit {limit} --max_tokens {max_tokens} '
               f'--tensor_parallel_size 1 --num_shards {num_gpus} --shard_id {g} --dtype float16')
        procs.append(subprocess.Popen(cmd, shell=True, env=env))
    for p in procs:
        assert p.wait() == 0, 'шард упал — см. лог выше'
    with open(out, 'w', encoding='utf-8') as fo:
        for sh in shards:
            with open(sh, encoding='utf-8') as fi:
                fo.write(fi.read())
    print('склеено →', out)

In [ ]:
# prepare_data.py читает ../dataset_ml_challenge.parquet и пишет
# data/{train_minimal,train_rich,eval}.jsonl + splits.json (детерм. held-out 800).
!python prepare_data.py
!wc -l data/train_minimal.jsonl data/eval.jsonl

## 2. SFT 1.7B (полный fp16 LoRA + DDP на 2×T4)

Loss только по токенам ответа (промпт замаскирован). 1.7B в fp16 (`--no_4bit`) легко влезает в 16GB T4 — полная точность базы лучше QLoRA для маленькой модели.
`--batch 4 --grad_accum 4` → эффективный батч 32 (16/карта × 2 карты), но кратно быстрее прежнего `batch=1`.

> Если OOM — снизь до `--batch 2 --grad_accum 8`. Если DDP капризничает — замени `accelerate launch ...` на `!python kaggle_sft.py ...` (1 карта).

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'  # меньше фрагментации
# QLoRA nf4 (4-bit база ~1.1ГБ вместо 3.4ГБ fp16) + batch 2 — надёжно влезает в 14.5ГБ T4.
# Эффективный батч 2*8*2карты=32. Если всё ещё OOM -> --batch 1 --grad_accum 16.
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 \
    kaggle_sft.py \
    --train_jsonl data/train_minimal.jsonl \
    --output_dir work/lora_1_7b \
    --base_model $BASE \
    --epochs 2 --max_len 2048 --neftune 5.0 \
    --batch 2 --grad_accum 8

In [ ]:
# Мерж адаптера в полные fp16-веса (vLLM грузит как обычную модель).
# На T4 (sm_75) merge_lora сохранит fp16 — ровно то, что ждёт рантайм (dtype=float16).
!python merge_lora.py --base_model $BASE --adapter work/lora_1_7b --out work/merged_1_7b
!du -sh work/merged_1_7b

## 3. Оффлайн-оценка (win-rate + тайминг)

Генерим ответы на held-out (800) точно как в рантайме (MINIMAL_SYSTEM, без few-shot, greedy), считаем win-rate против эталона локальным судьёй и смотрим экстраполяцию времени. NB: тайминг на T4 — верхняя оценка (L4 быстрее).

> Ориентир: win-rate > 0.5 = модель в среднем не хуже эталона. Сравни со стоковой base (запусти те же 2 ячейки с `--model_dir Qwen/Qwen3-1.7B`).

In [ ]:
# gen_candidates печатает tok/s, среднюю длину, % упёршихся в max_tokens и экстраполяцию на 4000.
!python gen_candidates.py \
    --model_dir work/merged_1_7b --variant minimal \
    --eval_jsonl data/eval.jsonl --out data/cand_1_7b.jsonl \
    --max_tokens 1280 --dtype float16

In [ ]:
!python judge_local.py --candidates data/cand_1_7b.jsonl \
    --judge_model $JUDGE_MODEL --dtype float16

## 4. Упаковка весов для посылки (FAST_SUBMIT)

Кладём fp16-веса в `weights/` посылки. Код посылки (`solution.py`, `source/`, `Dockerfile`) уже готов и в `config.py` выставлен `finetuned=True`. Скачай папку из Output и положи её содержимое в `EffectiveInference/weights/` перед сборкой zip.

In [ ]:
BEST = 'work/merged_1_7b_raft' if RUN_DISTILL else 'work/merged_1_7b'
PACK = '/kaggle/working/weights'
if os.path.exists(PACK):
    shutil.rmtree(PACK)
shutil.copytree(BEST, PACK)
!ls -la $PACK
!du -sh $PACK
print('\nГотово. Скачай /kaggle/working/weights из Output и положи его содержимое в weights/ посылки.')
print('Проверь: config.py finetuned=True; в weights/ есть config.json, tokenizer.json, *.safetensors.')

## 5. (ОПЦИОНАЛЬНО) Self-distill / RAFT поверх SFT

Запускай только если FAST вышел на плато (`RUN_DISTILL = True` в Config). Дороже (~1.5 ч) и рискованнее: локальный судья ≠ контестный → риск reward-hacking. Генерим кандидатов SFT-моделью (data-parallel), эвристики+судья отбирают «не ниже эталона», дообучаем с нуля от базы. Затем перезапусти ячейки 3–4 для оценки/упаковки `merged_1_7b_raft`.

In [ ]:
import json
if RUN_DISTILL:
    # 3a. Кандидаты SFT-моделью (fp16 merged), data-parallel на 2 карты.
    gen_distill_dp('work/merged_1_7b', 'data/distill_cand_1_7b.jsonl',
                   DISTILL_N, DISTILL_LIMIT, max_tokens=EVAL_MAX_TOKENS)
    # 3b. Отбор: эвристики → судья (TP=2). Печатает % замен эталона.
    !python select_distill.py \
        --candidates data/distill_cand_1_7b.jsonl \
        --out data/raft_1_7b.jsonl \
        --judge_model $JUDGE_MODEL --topk 2 --tensor_parallel_size 2
    # 3c. RAFT-набор (заменённые/подтверждённые) + хвост train_minimal, которого distill не касался.
    raft = [json.loads(l) for l in open('data/raft_1_7b.jsonl', encoding='utf-8')]
    tail = [json.loads(l) for l in open('data/train_minimal.jsonl', encoding='utf-8')][DISTILL_LIMIT:]
    with open('data/sft2_1_7b.jsonl', 'w', encoding='utf-8') as f:
        for r in raft + tail:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print('SFT-2 набор:', len(raft) + len(tail), 'примеров (raft', len(raft), '+ tail', len(tail), ')')
    # 3d. Дообучаем с нуля от базы на RAFT-наборе.
    subprocess.run('accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 '
        'kaggle_sft.py --train_jsonl data/sft2_1_7b.jsonl --output_dir work/lora_1_7b_raft '
        f'--base_model {BASE} --epochs 2 --max_len 2048 --neftune 5.0 --batch 2 --grad_accum 8',
        shell=True, check=True)
    subprocess.run(f'python merge_lora.py --base_model {BASE} --adapter work/lora_1_7b_raft --out work/merged_1_7b_raft',
        shell=True, check=True)
    print('\nRAFT готов. Перезапусти ячейки 3 (с --model_dir work/merged_1_7b_raft) и 4 для оценки/упаковки.')
else:
    print('RUN_DISTILL=False — пропущено (FAST_SUBMIT).')